# 08 · Late Delivery: Model Training and Selection

**Project:** Enterprise Supply Chain Intelligence Platform (DataCo)
**Task:** Binary classification of `Late_delivery_risk` using only information known at order placement.

This notebook answers one question: **which model should go forward to final evaluation?**

**Rules for this notebook**
- It uses the **train** and **validation** periods from notebook 07. The **test period is dropped on load** and is not touched until notebook 09.
- Every model is a scikit-learn `Pipeline`, so imputation, scaling and encoding are fitted on training data only.
- Simple baselines come first. A complex model must justify itself against them.
- Cross-validation respects time order (each fold validates on data later than it trained on).
- The final choice uses training evidence (cross-validation) and validation evidence (a later period), and prefers the simpler model when the difference is negligible.

In [1]:
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             precision_recall_curve, roc_auc_score)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.width', 140)

# ---- Paths (Windows): same locations as notebook 07 ----
PROJECT_ROOT = Path('E:/Supply Chain Project')
DATA_DIR   = PROJECT_ROOT / 'Datasets/Model Input'   # written by notebook 07
MODEL_DIR  = PROJECT_ROOT / 'Models'
FIG_DIR    = PROJECT_ROOT / 'Reports/Figures'
REPORT_DIR = PROJECT_ROOT / 'Reports'

assert (DATA_DIR / 'model_input.csv.gz').is_file(), 'Run notebook 07 first.'
for folder in (MODEL_DIR, FIG_DIR, REPORT_DIR):
    assert not folder.is_file(), f'{folder} is a file. It must be a folder.'
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
TUNE_TOP_K = 2        # how many model families get hyperparameter tuning
N_ITER     = 8        # random-search samples per family (raise for a deeper search)

def savefig(name):
    plt.savefig(FIG_DIR / f'{name}.png', dpi=150, bbox_inches='tight')

## 1. Load train and validation only

In [2]:
from datetime import datetime

# ---- Guard: make sure notebook 07 has been run and its outputs are current ----
csv_path = DATA_DIR / 'model_input.csv.gz'
man_path = DATA_DIR / 'feature_manifest.json'
for p in (csv_path, man_path):
    assert p.is_file(), f'Missing {p.name} in {DATA_DIR}. Run notebook 07 first.'

with open(man_path) as f:
    manifest = json.load(f)
header = pd.read_csv(csv_path, nrows=0).columns.tolist()

problems = []
if 'split' not in header:
    problems.append("model_input.csv.gz has no 'split' column")
if 'evaluation_boundaries' not in manifest:
    problems.append("feature_manifest.json has no 'evaluation_boundaries' entry")
if problems:
    stamp = datetime.fromtimestamp(csv_path.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
    raise RuntimeError(
        'Notebook 07 outputs are out of date (model_input.csv.gz was written ' + stamp + ').\n'
        + 'Problems: ' + '; '.join(problems) + '.\n'
        + 'Fix: open 07_late_delivery_data_audit_leakage_control.ipynb, choose Restart and Run All, '
        + 'check that its last cell prints "Verified", then run this notebook again.\n'
        + 'Folder checked: ' + str(DATA_DIR))

TARGET, DATE_COL = manifest['target'], manifest['date_col']
NUM, CAT = manifest['numeric'], manifest['categorical']
FEATURES = NUM + CAT

data = pd.read_csv(DATA_DIR / 'model_input.csv.gz', parse_dates=[DATE_COL])
data = data[data['split'].isin(['train', 'validation'])]      # the test period is deliberately dropped
data = data.sort_values(DATE_COL, kind='stable').reset_index(drop=True)

train = data[data['split'] == 'train']
val   = data[data['split'] == 'validation']
X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]

assert 'test' not in set(data['split']), 'Test rows must not be loaded in this notebook.'
print(f'{len(NUM)} numeric/flag features | {len(CAT)} categorical features')
pd.DataFrame(manifest['evaluation_boundaries']).T.loc[['train', 'validation']]

32 numeric/flag features | 7 categorical features


,rows,from,to,late_rate
train,122752,2015-01-01,2016-12-17,0.5491
validation,21664,2016-12-17,2017-04-22,0.5469


## 2. Preprocessing

Numeric features are median-imputed (and scaled for the linear model). Categorical features are mode-imputed and one-hot encoded. Levels below 1% frequency are grouped together, so a level never seen in training falls into that group at prediction time instead of raising an error.

In [3]:
def make_preprocessor(scale: bool) -> ColumnTransformer:
    num_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler() if scale else 'passthrough'),
    ])
    cat_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                 min_frequency=0.01, sparse_output=False)),
    ])
    transformers = [('num', num_pipe, NUM)]
    if CAT:
        transformers.append(('cat', cat_pipe, CAT))
    return ColumnTransformer(transformers)


def recall_at_precision(y_true, proba, target_precision=0.80):
    # Highest recall reachable while keeping precision at or above the target.
    p, r, _ = precision_recall_curve(y_true, proba)
    ok = p >= target_precision
    return float(r[ok].max()) if ok.any() else 0.0


def score_table(y_true, proba):
    return {
        'roc_auc':       roc_auc_score(y_true, proba),
        'pr_auc':        average_precision_score(y_true, proba),
        'brier':         brier_score_loss(y_true, proba),
        'recall@80%prec': recall_at_precision(y_true, proba, 0.80),
    }

## 3. Baselines

Two baselines set the bar:
1. **Majority class:** the same score for every order. It shows what "no model" looks like.
2. **Planner rule:** a logistic regression that sees only the scheduled shipping days and the shipping mode. An operations team could build it in a spreadsheet.

A complex model is only worth keeping if it clearly beats the second one.

In [4]:
rule_cols = [c for c in NUM if c.startswith('Shipping Mode_') or c == 'Days for shipment (scheduled)']
print('Planner-rule features:', rule_cols)

rule_pre = ColumnTransformer([('r', Pipeline([('i', SimpleImputer(strategy='median')),
                                              ('s', StandardScaler())]), rule_cols)])
planner_rule = Pipeline([('pre', rule_pre), ('clf', LogisticRegression(max_iter=1000))])
planner_rule.fit(X_train, y_train)

val_scores = {
    'Majority class': score_table(y_val, np.full(len(y_val), y_train.mean())),
    'Planner rule (mode + scheduled days)': score_table(y_val, planner_rule.predict_proba(X_val)[:, 1]),
}
pd.DataFrame(val_scores).T.round(4)

Planner-rule features: ['Days for shipment (scheduled)', 'Shipping Mode_Same Day', 'Shipping Mode_Second Class', 'Shipping Mode_Standard Class']


,roc_auc,pr_auc,brier,recall@80%prec
Majority class,0.5000,0.5469,0.2478,0.0000
Planner rule (mode + scheduled days),0.7264,0.7463,0.1974,0.5353


## 4. Candidate models and time-aware cross-validation

Three model families: a regularised linear model (interpretable), a random forest (non-linear, robust), and histogram gradient boosting (usually the strongest on tabular data). Cross-validation uses `TimeSeriesSplit` on the training period only.

In [5]:
candidates = {
    'Logistic Regression': Pipeline([
        ('pre', make_preprocessor(True)),
        ('clf', LogisticRegression(max_iter=2000, C=1.0))]),
    'Random Forest': Pipeline([
        ('pre', make_preprocessor(False)),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=20,
                                       n_jobs=-1, random_state=SEED))]),
    'Hist Gradient Boosting': Pipeline([
        ('pre', make_preprocessor(False)),
        ('clf', HistGradientBoostingClassifier(learning_rate=0.1, max_iter=300,
                                               early_stopping=True, random_state=SEED))]),
}

tscv = TimeSeriesSplit(n_splits=4)
cv_rows = {}
for name, pipe in candidates.items():
    t0 = time.time()
    res = cross_validate(pipe, X_train, y_train, cv=tscv,
                         scoring={'roc_auc': 'roc_auc', 'pr_auc': 'average_precision'}, n_jobs=1)
    cv_rows[name] = {'cv_roc_auc': res['test_roc_auc'].mean(),
                     'cv_roc_auc_std': res['test_roc_auc'].std(),
                     'cv_pr_auc': res['test_pr_auc'].mean(),
                     'seconds': time.time() - t0}
    print(f'{name:<24} CV ROC-AUC {cv_rows[name]["cv_roc_auc"]:.4f} '
          f'(+/- {cv_rows[name]["cv_roc_auc_std"]:.4f})   [{cv_rows[name]["seconds"]:.0f}s]')

cv_table = pd.DataFrame(cv_rows).T.sort_values('cv_roc_auc', ascending=False).round(4)
cv_table

Logistic Regression      CV ROC-AUC 0.7409 (+/- 0.0050)   [11s]
Random Forest            CV ROC-AUC 0.7397 (+/- 0.0048)   [67s]
Hist Gradient Boosting   CV ROC-AUC 0.7326 (+/- 0.0067)   [79s]


,cv_roc_auc,cv_roc_auc_std,cv_pr_auc,seconds
Logistic Regression,0.7409,0.0050,0.8165,11.3427
Random Forest,0.7397,0.0048,0.8146,67.2993
Hist Gradient Boosting,0.7326,0.0067,0.8096,79.0858


## 5. Hyperparameter tuning

The top families from cross-validation get a small randomised search, using the same time-ordered folds. The search is deliberately modest. The goal is a sensible, regularised configuration, not the last 0.001 of AUC. The family that is not tuned stays at its defaults and is still compared fairly on the validation period in the next section.

In [ ]:
param_spaces = {
    'Hist Gradient Boosting': {
        'clf__learning_rate':     [0.03, 0.05, 0.08, 0.1],
        'clf__max_leaf_nodes':    [15, 31, 63],
        'clf__min_samples_leaf':  [20, 50, 100, 200],
        'clf__l2_regularization': [0.0, 0.1, 1.0, 5.0],
        'clf__max_iter':          [200, 400],
    },
    'Random Forest': {
        'clf__max_depth':        [8, 12, 16, None],
        'clf__min_samples_leaf': [10, 20, 50],
        'clf__max_features':     ['sqrt', 0.3, 0.5],
        'clf__n_estimators':     [150, 250],
    },
    'Logistic Regression': {'clf__C': [0.01, 0.1, 1.0, 10.0]},
}

fitted, tuned_info = {}, {}
families_to_tune = cv_table.index[:TUNE_TOP_K].tolist()
print('Tuning:', families_to_tune)

for name in candidates:
    if name in families_to_tune:
        space = param_spaces[name]
        n_iter = min(N_ITER, int(np.prod([len(v) for v in space.values()])))
        t0 = time.time()
        search = RandomizedSearchCV(candidates[name], space, n_iter=n_iter, cv=tscv,
                                    scoring='roc_auc', random_state=SEED, n_jobs=1, refit=True)
        search.fit(X_train, y_train)
        fitted[name + ' (tuned)'] = search.best_estimator_
        tuned_info[name + ' (tuned)'] = {'cv_roc_auc': round(search.best_score_, 4),
                                         'best_params': search.best_params_}
        print(f'{name}: best CV ROC-AUC {search.best_score_:.4f}  [{time.time()-t0:.0f}s]  {search.best_params_}')
    else:
        fitted[name] = candidates[name].fit(X_train, y_train)
        tuned_info[name] = {'cv_roc_auc': cv_table.loc[name, 'cv_roc_auc'], 'best_params': 'defaults'}

Tuning: ['Logistic Regression', 'Random Forest']
Logistic Regression: best CV ROC-AUC 0.7409  [66s]  {'clf__C': 10.0}


## 6. Comparing everything on the validation period

The validation period comes after the training period in time, so this is the closest stand-in for how each model would behave on new orders. Three kinds of evidence go into the table:

- **Train ROC-AUC vs validation ROC-AUC:** a large gap means the model memorised the training period
- **Cross-validation ROC-AUC:** how stable the model was across earlier folds
- **Validation metrics:** ROC-AUC and PR-AUC (ranking quality), Brier score (probability quality) and recall at 80% precision (usefulness when flagged orders must be mostly right)

In [ ]:
rows = {}
for name, pipe in fitted.items():
    p_tr  = pipe.predict_proba(X_train)[:, 1]
    p_val = pipe.predict_proba(X_val)[:, 1]
    s = score_table(y_val, p_val)
    rows[name] = {
        'train_roc_auc': roc_auc_score(y_train, p_tr),
        'cv_roc_auc':    tuned_info[name]['cv_roc_auc'],
        'val_roc_auc':   s['roc_auc'],
        'val_pr_auc':    s['pr_auc'],
        'val_brier':     s['brier'],
        'val_recall@80%prec': s['recall@80%prec'],
    }
sel = pd.DataFrame(rows).T
sel['overfit_gap'] = sel['train_roc_auc'] - sel['val_roc_auc']
sel = sel.sort_values('val_roc_auc', ascending=False)

rule_auc = val_scores['Planner rule (mode + scheduled days)']['roc_auc']
sel['gain_vs_planner_rule'] = sel['val_roc_auc'] - rule_auc
sel.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
order = sel.index.tolist()
x = np.arange(len(order)); w = 0.27
ax[0].bar(x - w, sel['train_roc_auc'], w, label='Train', color='#8ecae6')
ax[0].bar(x,     sel['cv_roc_auc'],    w, label='Cross-val', color='#219ebc')
ax[0].bar(x + w, sel['val_roc_auc'],   w, label='Validation', color='#023047')
ax[0].axhline(rule_auc, color='#e76f51', ls='--', label='Planner rule (validation)')
ax[0].set_xticks(x); ax[0].set_xticklabels(order, rotation=20, ha='right')
ax[0].set_ylim(max(0.5, sel[['train_roc_auc', 'cv_roc_auc', 'val_roc_auc']].min().min() - 0.05), 1.0)
ax[0].set_title('ROC-AUC: train vs cross-validation vs validation'); ax[0].legend()

ax[1].barh(order[::-1], sel['val_recall@80%prec'].iloc[::-1], color='#2a9d8f')
ax[1].set_title('Recall at 80% precision (validation)')
ax[1].set_xlabel('Share of late orders caught while keeping flags 80% correct')
plt.tight_layout(); savefig('07_model_selection'); plt.show()

## 7. Selecting the model

The rule is written down before looking at the winner:

1. Rank the candidates by validation ROC-AUC.
2. Any model within **0.002 ROC-AUC** of the best counts as tied. Among tied models, choose the **simplest** (logistic regression, then random forest or gradient boosting). A tiny gain is not worth extra complexity.
3. Check the chosen model against the planner rule. If it does not beat the rule by at least **0.005 ROC-AUC**, say so plainly. In that case a rule-based approach is the better recommendation.

In [ ]:
TIE_TOL, MIN_GAIN = 0.002, 0.005
complexity = {'Logistic Regression': 1, 'Random Forest': 2, 'Hist Gradient Boosting': 2}
sel['complexity'] = [complexity[n.replace(' (tuned)', '')] for n in sel.index]

best_auc = sel['val_roc_auc'].max()
tied = sel[sel['val_roc_auc'] >= best_auc - TIE_TOL]
selected_name = tied.sort_values(['complexity', 'val_roc_auc'], ascending=[True, False]).index[0]
selected = fitted[selected_name]

print(f'Highest validation ROC-AUC: {sel["val_roc_auc"].idxmax()} ({best_auc:.4f})')
print(f'Models within {TIE_TOL} of the best: {tied.index.tolist()}')
print(f'\nSELECTED: {selected_name}')
gain = sel.loc[selected_name, 'val_roc_auc'] - rule_auc
print(f'Validation ROC-AUC gain over the planner rule: {gain:+.4f}')
if gain < MIN_GAIN:
    print(f'WARNING: gain is below {MIN_GAIN}. The ML model does not clearly beat the simple rule on this data.')
else:
    print(f'The selected model beats the planner rule by more than {MIN_GAIN}.')
print('Overfit gap (train - validation ROC-AUC): ', round(sel.loc[selected_name, 'overfit_gap'], 4))

## 8. Save the selected model for evaluation

The model saved here was fitted on the training period only. It carries no decision threshold yet. Choosing the threshold and the final test evaluation both happen in notebook 09.

In [ ]:
artifact = {
    'pipeline': selected,
    'selected_name': selected_name,
    'family': selected_name.replace(' (tuned)', ''),
    'best_params': tuned_info[selected_name]['best_params'],
    'features': FEATURES,
    'numeric_features': NUM,
    'categorical_features': CAT,
    'target': TARGET,
    'validation_metrics': {k: float(v) for k, v in sel.loc[selected_name].drop('complexity').items()},
    'planner_rule_val_roc_auc': float(rule_auc),
    'sklearn_version': sklearn.__version__,
    'trained_on': manifest['evaluation_boundaries']['train'],
}
joblib.dump(artifact, MODEL_DIR / 'selected_model.joblib')

sel.drop(columns='complexity').round(4).to_csv(REPORT_DIR / 'model_selection.csv')
cv_table.to_csv(REPORT_DIR / 'cross_validation.csv')
print('Saved:', MODEL_DIR / 'selected_model.joblib')
print('Saved:', REPORT_DIR / 'model_selection.csv')

## Summary

- Two baselines (majority class and a shipping-mode rule) were scored on the validation period, and three model families were cross-validated with time-ordered folds.
- The top families were tuned with a modest randomised search.
- Candidates were compared on train, cross-validation and validation evidence, including an overfitting check.
- The model was chosen by a rule written before looking at results, which prefers the simpler model on near-ties and reports honestly whether ML beats the simple rule.
- The test period was not touched. Next: `09_late_delivery_evaluation_business_impact.ipynb`.